In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import norm
import sys
sys.path.append('../helpers')
import constants as vp
from converter import convert_sd_to_EN

track_df = pd.read_csv("../data/track_centerline.csv")

kappa_abs = np.abs(track_df['kappa_c'].values)
KAPPA_MIN = np.min(kappa_abs)
KAPPA_MAX = np.max(kappa_abs)

def curvature_factor(kappa):
    if KAPPA_MAX == KAPPA_MIN:
        return 0.0
    kappa_abs_val = abs(kappa)
    if kappa_abs_val < KAPPA_MIN:
        return 0.0
    elif kappa_abs_val > KAPPA_MAX:
        return 1.0
    else:
        return (kappa_abs_val - KAPPA_MIN) / (KAPPA_MAX - KAPPA_MIN)

def dynamic_s_step(kappa):
    return vp.DS_MAX - curvature_factor(kappa) * (vp.DS_MAX - vp.DS_MIN)

s_vals = track_df['s'].values
kappa_vals = track_df['kappa_c'].values
s_max = s_vals[-1]

s_current = s_vals[0]
sampled_indices = []

while s_current < s_max:
    idx = min(np.searchsorted(s_vals, s_current), len(s_vals) - 1)
    sampled_indices.append(idx)
    s_current += dynamic_s_step(kappa_vals[idx])

sampled_indices.append(len(s_vals) - 1)
sampled_indices = sorted(list(set(sampled_indices)))
sampled_s_vals = s_vals[sampled_indices]

s_diffs = np.diff(sampled_s_vals)
diff_df = pd.DataFrame({
    's_start': sampled_s_vals[:-1],
    's_end': sampled_s_vals[1:],
    's_diff': np.diff(sampled_s_vals)
})

velocity_profile_df = pd.DataFrame({
    's': sampled_s_vals,
    'idx': sampled_indices,
    'kappa': kappa_vals[sampled_indices]
})

v_turn = np.sqrt(vp.MU * vp.G / np.abs(velocity_profile_df['kappa'].replace(0, np.nan))).fillna(0.0)
v_turn.iloc[0] = 0.0
v_turn.iloc[-1] = 0.0
velocity_profile_df['v_turn'] = v_turn

v_ref_vals = velocity_profile_df['v_turn'].to_numpy()
s_vals = velocity_profile_df['s'].to_numpy()

minima_indices = [0]
n = len(v_ref_vals)
for i in range(1, n - 1):
    if v_ref_vals[i] < v_ref_vals[i - 1] and v_ref_vals[i] <= v_ref_vals[i + 1]:
        minima_indices.append(i)
if n > 1 and v_ref_vals[-1] < v_ref_vals[-2]:
    minima_indices.append(n - 1)

mptr = 0
a = 0.8

for i in range(1, n):
    while mptr + 1 < len(minima_indices) and minima_indices[mptr + 1] < i:
        mptr += 1
    recent_min_idx = minima_indices[mptr]
    v_acc = np.sqrt(v_ref_vals[recent_min_idx] ** 2 + 2 * a * (s_vals[i] - s_vals[recent_min_idx]))
    if v_acc < v_ref_vals[i]:
        v_ref_vals[i] = v_acc

velocity_profile_df['v_ref'] = v_ref_vals

S_rem = vp.TOTAL_TRACK_LENGTH
T_rem = vp.TIME_HORIZON

velocity_profile_df = velocity_profile_df.merge(
    diff_df[['s_start', 's_diff']],
    how='left',
    left_on='s',
    right_on='s_start'
)
velocity_profile_df['s_diff'] = velocity_profile_df['s_diff'].bfill()
velocity_profile_df = velocity_profile_df.drop(columns=['s_start'])

valid_mask = (velocity_profile_df['v_ref'] > 0)
sorted_df = velocity_profile_df[valid_mask].sort_values(by='v_ref', ascending=True).copy()

S_rem = vp.TOTAL_TRACK_LENGTH
T_rem = vp.TIME_HORIZON

region_labels = []
inter_loci_mode = False

for idx, row in sorted_df.iterrows():
    v_ref_i = row['v_ref']
    ds_i = row['s_diff']
    
    v_avg = S_rem / T_rem
    
    if not inter_loci_mode:
        if v_ref_i < v_avg:
            region_labels.append("loci")
            S_rem -= ds_i
            T_rem -= ds_i / v_ref_i
        else:
            inter_loci_mode = True
            region_labels.append("inter-loci")
    else:
        region_labels.append("inter-loci")

sorted_df['region'] = region_labels

velocity_profile_df = velocity_profile_df.merge(
    sorted_df[['s', 'region']],
    on='s',
    how='left'
)

velocity_profile_df['region'] = velocity_profile_df['region'].fillna('loci')
velocity_profile_df.loc[velocity_profile_df.index[0], 'v_ref'] = 0.0
velocity_profile_df.loc[velocity_profile_df.index[-1], 'v_ref'] = 0.0

print(velocity_profile_df)

               s  idx     kappa     v_turn      v_ref     s_diff      region
0       0.000000    0 -0.001954   0.000000   0.000000  36.797220        loci
1      36.797220    2  0.000621   7.673041   7.673041  29.842560  inter-loci
2      66.639779    3  0.000709  10.325873  10.325873  31.993050  inter-loci
3      98.632829    4  0.000389  12.562346  12.562346  29.498140  inter-loci
4     128.130970    5 -0.000106  14.318155  14.318155  13.291061  inter-loci
..           ...  ...       ...        ...        ...        ...         ...
183  3624.040163  393  0.000010  30.818440  30.818440  15.973916  inter-loci
184  3640.014079  395 -0.009963  22.188165  22.188165  24.243334  inter-loci
185  3664.257413  398 -0.012143  20.098422  20.098422  13.774213  inter-loci
186  3678.031626  400  0.016360  17.315385  17.315385  12.012145  inter-loci
187  3690.043771  403  0.156429   0.000000   0.000000        NaN        loci

[188 rows x 7 columns]


In [2]:
velocity_profile_df = velocity_profile_df.sort_values(by='s').reset_index(drop=True)

regions = velocity_profile_df['region'].to_numpy()
n = len(regions)

is_loci = regions == 'loci'
is_inter = regions == 'inter-loci'

boundary_mask = np.zeros(n, dtype=bool)

for i in range(n):
    prev_region = regions[i-1] if i > 0 else None
    next_region = regions[i+1] if i < n-1 else None

    if ((regions[i] == 'loci' and next_region == 'inter-loci') or
        (regions[i] == 'loci' and prev_region == 'inter-loci')):
        boundary_mask[i] = True

velocity_profile_df.loc[boundary_mask, 'region'] = 'boundary'

regions = velocity_profile_df['region'].to_numpy()
s_vals = velocity_profile_df['s'].to_numpy()

interloci_indices = np.where(regions == 'inter-loci')[0]
boundary_indices = np.where(regions == 'boundary')[0]

interloci_groups = []
if len(interloci_indices) > 0:
    current_group = [interloci_indices[0]]
    for idx in interloci_indices[1:]:
        if idx == current_group[-1] + 1:
            current_group.append(idx)
        else:
            interloci_groups.append(current_group)
            current_group = [idx]
    interloci_groups.append(current_group)

boundary_pairs = []

for group in interloci_groups:
    start_idx, end_idx = group[0], group[-1]

    left_candidates = boundary_indices[boundary_indices < start_idx]
    left_boundary = left_candidates[-1] if len(left_candidates) > 0 else None

    right_candidates = boundary_indices[boundary_indices > end_idx]
    right_boundary = right_candidates[0] if len(right_candidates) > 0 else None

    if left_boundary is not None and right_boundary is not None:
        boundary_pairs.append((left_boundary, right_boundary))
    elif left_boundary is not None:
        boundary_pairs.append((left_boundary, None))
    elif right_boundary is not None:
        boundary_pairs.append((None, right_boundary))
    else:
        boundary_pairs.append((None, None))

boundary_tuples = []
for left_idx, right_idx in boundary_pairs:
    s_left = s_vals[left_idx] if left_idx is not None else None
    s_right = s_vals[right_idx] if right_idx is not None else None
    boundary_tuples.append((s_left, s_right))

print("Updated regions with boundaries:")
print(velocity_profile_df[['s', 'region']])

print("\nBoundary tuples enclosing inter-loci regions:")
for b in boundary_tuples:
    print(b)


Updated regions with boundaries:
               s      region
0       0.000000    boundary
1      36.797220  inter-loci
2      66.639779  inter-loci
3      98.632829  inter-loci
4     128.130970  inter-loci
..           ...         ...
183  3624.040163  inter-loci
184  3640.014079  inter-loci
185  3664.257413  inter-loci
186  3678.031626  inter-loci
187  3690.043771    boundary

[188 rows x 2 columns]

Boundary tuples enclosing inter-loci regions:
(np.float64(0.0), np.float64(3690.043770792676))


In [3]:
for s1, s2 in boundary_tuples:
    D = s2 - s1
    T = T_rem * D / S_rem

    start_row = velocity_profile_df[(velocity_profile_df['s'] == s1) & (velocity_profile_df['region'] == 'boundary')]
    if start_row.empty:
        raise ValueError("Starting boundary point not found.")

    v_i = float(start_row['v_ref'].iloc[0])

    A = 1
    B = -(2*a*T + 2*v_i)
    C = v_i**2 + 2*a*D
    disc = B*B - 4*A*C
    if disc < 0:
        raise ValueError("No real roots for vc.")

    r1 = (-B + np.sqrt(disc)) / (2*A)
    r2 = (-B - np.sqrt(disc)) / (2*A)
    roots = [r for r in [r1, r2] if r > 0]
    if not roots:
        raise ValueError("No positive root for vc.")

    v_c = min(roots)
    print(f"vc for section {s1}–{s2}: {v_c}")

    mask = (
        (velocity_profile_df['s'] > s1) &
        (velocity_profile_df['s'] < s2) &
        (velocity_profile_df['region'] == 'inter-loci')
    )
    idxs = velocity_profile_df.index[mask]

    print(f"Matched rows: {len(idxs)}")

    velocity_profile_df.loc[idxs, 'v_ref'] = np.where(
        velocity_profile_df.loc[idxs, 'v_ref'] > v_c,
        v_c,
        velocity_profile_df.loc[idxs, 'v_ref']
    )

    velocity_profile_df.loc[idxs, 'region'] = 'updated'
    if len(idxs) > 0:
        print(f"Updated region count: {(velocity_profile_df.loc[idxs, 'region'] == 'updated').sum()}")


vc for section 0.0–3690.043770792676: 7.05806265100091
Matched rows: 186
Updated region count: 186


In [4]:
import plotly.graph_objects as go
import pandas as pd

track_df = pd.read_csv("../data/track_centerline.csv")

all_s = track_df['s'].values
all_E = track_df['E'].values
all_N = track_df['N'].values

sampled_E = np.interp(velocity_profile_df['s'], all_s, all_E)
sampled_N = np.interp(velocity_profile_df['s'], all_s, all_N)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=all_E,
    y=all_N,
    mode='lines',
    line=dict(color='lightgray', width=2),
    opacity=0.5
))

types = velocity_profile_df['region']

marker_sizes = np.where(types == 'interloci', 5, 8)
marker_lines = np.where(types == 'boundary', 2, 0)

fig.add_trace(go.Scatter(
    x=sampled_E,
    y=sampled_N,
    mode='markers',
    marker=dict(
        color=velocity_profile_df['v_ref'],
        size=marker_sizes,
        colorscale='bluered',
        showscale=True,
        colorbar=dict(title='Reference Velocity'),
        line=dict(width=marker_lines, color='black'),
        symbol='circle'
    ),
    text=[
        f"idx={i}<br>s={s:.2f}<br>v_ref={v:.2f}<br>type={t}"
        for i, s, v, t in zip(
            velocity_profile_df['idx'],
            velocity_profile_df['s'],
            velocity_profile_df['v_ref'],
            velocity_profile_df['region']
        )
    ]
))

fig.update_layout(
    xaxis_title='Easting',
    yaxis_title='Northing',
    template='plotly_dark',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    showlegend=False
)

fig.show()


In [5]:
def curvature_limited_velocity(kappa):
    return np.sqrt(vp.MU * vp.G / max(abs(kappa), 1e-6))

def dynamic_d_bins(kappa):
    f = curvature_factor(kappa)
    n = int(np.round(vp.N_D_MIN + f * (vp.N_D_MAX - vp.N_D_MIN)))
    return np.linspace(-vp.BOUNDARY_WIDTH, vp.BOUNDARY_WIDTH, 2 * n + 1)

def adjusted_kappa(kappa, d):
    r = 1.0 / max(abs(kappa), 1e-6)
    r_eff = r - d * np.sign(kappa)
    r_eff = max(abs(r_eff), 1e-3)
    return 1.0 / r_eff

def dynamic_v_bins(kappa, d, s):
    mask = np.isclose(velocity_profile_df['s'].to_numpy(), s)
    if not mask.any():
        # fallback: choose nearest s
        idx_near = np.argmin(np.abs(velocity_profile_df['s'].to_numpy() - s))
        row = velocity_profile_df.iloc[idx_near]
    else:
        row = velocity_profile_df.loc[mask].iloc[0]

    v_ref = row['v_ref']
    v_turn = row['v_turn']
    region = row['region']
    n_v = vp.N_V
    std_dev = vp.STD_DEV_V

    if (v_ref == 0) and (s == velocity_profile_df['s'].iloc[0] or s == velocity_profile_df['s'].iloc[-1]):
        return np.array([0.0])

    if region == 'updated':
        v_center = v_ref
        gaussian_offsets = norm.ppf(np.linspace(0.3, 0.6, n_v)) * std_dev * v_center

    elif region in ['loci', 'boundary'] and np.isclose(v_turn, v_ref):
        kappa_eff = adjusted_kappa(kappa, d)
        new_v = curvature_limited_velocity(kappa_eff)
        v_center = new_v if new_v < v_ref else v_ref
        quantiles = np.linspace(0.05, 0.5, n_v)
        gaussian_offsets = norm.ppf(quantiles) * std_dev * v_center

    elif region in ['loci', 'boundary'] and not np.isclose(v_turn, v_ref):
        v_center = v_ref
        quantiles = np.linspace(0.05, 0.5, n_v)
        gaussian_offsets = norm.ppf(quantiles) * std_dev * v_center

    else:
        v_center = max(v_ref, 0.0)
        quantiles = np.linspace(0.05, 0.5, n_v)
        gaussian_offsets = norm.ppf(quantiles) * std_dev * max(v_center, 1e-3)

    v_bins = v_center + gaussian_offsets
    v_bins = np.append(v_bins, v_center)
    v_bins = np.clip(v_bins, 0, None)
    v_bins = np.sort(v_bins)
    return np.unique(np.round(v_bins, 3))

def add_state(state_list, state_id, s, d, v):
    state_list.append({'state_id': state_id, 's': s, 'd': d, 'v': v})
    return state_id + 1

s_full = track_df['s'].values
kappa_full = track_df['kappa_c'].values

state_records = []
state_id_counter = 0

start_s = s_full[sampled_indices[0]]
state_id_counter = add_state(state_records, state_id_counter, start_s, 0.0, 0.0)

for idx in tqdm(sampled_indices[1:-1], desc="Sampling Frenet States"):
    s = s_full[idx]
    kappa = kappa_full[idx]

    d_bins = dynamic_d_bins(np.abs(kappa))

    for d in d_bins:
        v_bins = dynamic_v_bins(kappa, d, s)
        for v in v_bins:
            state_id_counter = add_state(state_records, state_id_counter, s, float(d), float(v))

end_s = s_full[sampled_indices[-1]]
state_id_counter = add_state(state_records, state_id_counter, end_s, 0.0, 0.0)

states_df = pd.DataFrame(state_records)
converted_df = convert_sd_to_EN(states_df)

converted_df.to_parquet("../data/frenet_states.parquet", engine='pyarrow', index=False)
print("Converted frenet states saved.")

Sampling Frenet States: 100%|██████████| 186/186 [00:00<00:00, 1111.58it/s]


Converted frenet states saved.


In [6]:
velocity_lookup = (
    velocity_profile_df[['s', 's_diff', 'v_ref']]
    .rename(columns={'v_ref': 'v'})
    .sort_values('s')
    .reset_index(drop=True)
)

velocity_lookup.to_csv("../data/velocity_lookup.csv", index=False)

temp = velocity_lookup.copy()
temp = temp[temp['v'] > 0]

temp['t'] = temp['s_diff'] / temp['v']
total_time = temp['t'].sum()

print(f"Total estimated time: {total_time:.3f} seconds")

Total estimated time: 517.599 seconds


In [7]:
import plotly.graph_objects as go

converted_df = pd.read_parquet("../data/frenet_states.parquet")
track_df = pd.read_csv("../data/track_centerline.csv")

all_s = track_df['s'].to_numpy()
all_E = track_df['E'].to_numpy()
all_N = track_df['N'].to_numpy()

sampled_s = np.sort(converted_df['s'].unique())
sampled_E = np.interp(sampled_s, all_s, all_E)
sampled_N = np.interp(sampled_s, all_s, all_N)
sampled_num_states = (
    converted_df.groupby('s').size()
    .reindex(sampled_s)
    .fillna(0)
    .to_numpy()
)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=all_E,
    y=all_N,
    mode='lines',
    line=dict(color='lightgray', width=2),
    opacity=0.5
))
fig1.add_trace(go.Scatter(
    x=sampled_E,
    y=sampled_N,
    mode='markers',
    marker=dict(
        color=sampled_num_states,
        colorbar=dict(title='Number of States'),
        size=8,
        colorscale='bluered',
        showscale=True,
        symbol='circle'
    ),
    text=[f's={s:.2f}<br>states={n}' for s, n in zip(sampled_s, sampled_num_states)],
    hovertemplate='E=%{x:.1f}<br>N=%{y:.1f}<br>%{text}<extra></extra>'
))
fig1.update_layout(
    title='Adaptive Binning for Discretization',
    xaxis_title='Easting',
    yaxis_title='Northing',
    template='plotly_dark',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    showlegend=False
)
fig1.show()

en_group = (
    converted_df.groupby(['E', 'N'])['v']
    .max()
    .reset_index()
    .sort_values(['E', 'N'])
)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=all_E,
    y=all_N,
    mode='lines',
    line=dict(color='lightgray', width=2),
    opacity=0.5
))
fig2.add_trace(go.Scatter(
    x=en_group['E'],
    y=en_group['N'],
    mode='markers',
    marker=dict(
        color=en_group['v'],
        colorbar=dict(title='Max Velocity'),
        size=8,
        colorscale='bluered',
        showscale=True,
        symbol='circle'
    ),
    text=[f'v_max={v:.2f}' for v in en_group['v']],
    hovertemplate='E=%{x:.1f}<br>N=%{y:.1f}<br>%{text}<extra></extra>'
))
fig2.update_layout(
    title='Max Velocity in EN Space',
    xaxis_title='Easting',
    yaxis_title='Northing',
    template='plotly_dark',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    showlegend=False
)
fig2.show()